# Anomaly Detection Training


This notebook trains anomaly detection models to identify unusual electricity demand patterns.

## Methods Used:
- **Isolation Forest**: Unsupervised anomaly detection with contamination=0.03 (300 trees). This is the final anomaly flag.
- **Z-Score**: Statistical check using threshold=3.0, kept as a diagnostic only (it never reaches 3.0 on this data)
- **Final flag**: Isolation Forest only, the same as `backend/app/ml/anomaly_detection.py` (the earlier OR-combination was removed)

## Steps:
1. Load daily profiles and clustering results (notebooks/outputs/, made by notebooks 02 and 03)
2. Load profile_scaler.pkl (from notebook 03) and scale features
3. Train Isolation Forest and save to notebooks/outputs/models/isolation_forest_model.pkl
4. Calculate Z-Score diagnostics
5. Use the Isolation Forest flag as the final anomaly flag and create anomaly reasons
6. Save anomaly results and metrics to notebooks/outputs/


> **Note (19 Oct 2026): this notebook saves its files to `notebooks/outputs/`, not `data/processed/` or `models/`.**
>
> The dashboard and API read `data/processed/` and `models/`. So that running this notebook can never overwrite those production files, every file it saves goes to `notebooks/outputs/` instead (models to `notebooks/outputs/models/`). Notebooks 01 → 05 read each other's files from that folder, so the chain still works end to end.
>
> The logic is the same as the app modules in `backend/app/ml/`. The last cell compares this notebook's results with the production files (read-only).

In [1]:

import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import sys
from scipy import stats

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import sklearn components
from sklearn.ensemble import IsolationForest

# Set up paths
# Outputs go to notebooks/outputs/ so production data/processed/ and models/ are never overwritten
DATA_PROCESSED_DIR = PROJECT_ROOT / "notebooks" / "outputs"
MODELS_DIR = DATA_PROCESSED_DIR / "models"
DAILY_PROFILES_PATH = DATA_PROCESSED_DIR / "daily_profiles.csv"
CLUSTERING_RESULTS_PATH = DATA_PROCESSED_DIR / "clustering_results.csv"
ANOMALY_RESULTS_PATH = DATA_PROCESSED_DIR / "anomaly_results.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Daily profiles path: {DAILY_PROFILES_PATH}")
print(f"Clustering results path: {CLUSTERING_RESULTS_PATH}")


Project root: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence
Daily profiles path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\daily_profiles.csv
Clustering results path: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\clustering_results.csv


In [2]:

# Load daily profiles
if not DAILY_PROFILES_PATH.exists():
    raise FileNotFoundError(f"Daily profiles not found at {DAILY_PROFILES_PATH}")

daily_profiles = pd.read_csv(DAILY_PROFILES_PATH)
print(f"Daily profiles loaded: {daily_profiles.shape}")

# Load clustering results
if not CLUSTERING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Clustering results not found at {CLUSTERING_RESULTS_PATH}")

clustering_results = pd.read_csv(CLUSTERING_RESULTS_PATH)
print(f"Clustering results loaded: {clustering_results.shape}")

# Load profile scaler
scaler_path = MODELS_DIR / "profile_scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found at {scaler_path}")

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)
print(f"Scaler loaded successfully")


Daily profiles loaded: (1977, 106)
Clustering results loaded: (1977, 10)
Scaler loaded successfully


In [3]:

# Prepare features for anomaly detection
profile_cols = [f"p{str(i).zfill(2)}" for i in range(96)]
available_cols = [col for col in profile_cols if col in daily_profiles.columns]
X = daily_profiles[available_cols].values

print(f"Features prepared:")
print(f"- Number of samples (days): {X.shape[0]}")
print(f"- Number of features (time slots): {X.shape[1]}")

# Scale features using the same scaler from clustering
X_scaled = scaler.transform(X)

print(f"Features standardized using existing scaler")
print(f"Scaled data shape: {X_scaled.shape}")


Features prepared:
- Number of samples (days): 1977
- Number of features (time slots): 96
Features standardized using existing scaler
Scaled data shape: (1977, 96)


In [4]:

# Train Isolation Forest model
print("Training Isolation Forest model...")

iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.03,  # Expect 3% anomalies
    random_state=42
)

iso_forest.fit(X_scaled)

# Calculate anomaly scores and predictions
iso_anomaly_scores = -iso_forest.decision_function(X_scaled)  # Higher = more anomalous
iso_anomaly_flags = iso_forest.predict(X_scaled) == -1  # -1 indicates anomaly

print(f"Isolation Forest trained successfully!")
print(f"Anomalies detected: {iso_anomaly_flags.sum()} ({iso_anomaly_flags.mean()*100:.2f}% of data)")
print(f"Expected contamination: 3.0%")
print(f"Actual contamination: {iso_anomaly_flags.mean()*100:.2f}%")

# Save Isolation Forest model
with open(MODELS_DIR / "isolation_forest_model.pkl", "wb") as f:
    pickle.dump(iso_forest, f)

print(f"Isolation Forest model saved to: {MODELS_DIR / 'isolation_forest_model.pkl'}")


Training Isolation Forest model...


Isolation Forest trained successfully!
Anomalies detected: 60 (3.03% of data)
Expected contamination: 3.0%
Actual contamination: 3.03%
Isolation Forest model saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\isolation_forest_model.pkl


In [5]:

# Calculate Z-Score based anomaly detection
print("Calculating Z-Score based anomaly detection...")

# Calculate Z-Scores for daily statistics
peak_z_scores = np.abs(stats.zscore(daily_profiles['daily_peak'], nan_policy='omit'))
mean_z_scores = np.abs(stats.zscore(daily_profiles['daily_mean'], nan_policy='omit'))

# Define anomaly thresholds
Z_SCORE_THRESHOLD = 3.0

# Create anomaly flags
peak_z_anomaly = peak_z_scores >= Z_SCORE_THRESHOLD
mean_z_anomaly = mean_z_scores >= Z_SCORE_THRESHOLD

# Combined Z-score anomaly flag
z_score_anomaly_flags = peak_z_anomaly | mean_z_anomaly

print(f"Z-Score anomaly detection completed!")
print(f"Peak demand anomalies: {peak_z_anomaly.sum()} ({peak_z_anomaly.mean()*100:.2f}%)")
print(f"Mean demand anomalies: {mean_z_anomaly.sum()} ({mean_z_anomaly.mean()*100:.2f}%)")
print(f"Combined Z-score anomalies: {z_score_anomaly_flags.sum()} ({z_score_anomaly_flags.mean()*100:.2f}%)")


Calculating Z-Score based anomaly detection...
Z-Score anomaly detection completed!
Peak demand anomalies: 0 (0.00%)
Mean demand anomalies: 0 (0.00%)
Combined Z-score anomalies: 0 (0.00%)


In [6]:

# Final anomaly flag = Isolation Forest only (same as the app).
# The z-score never reaches 3.0 on this data, so it is kept as a diagnostic, not combined.
print("Setting the final anomaly flag...")

final_anomaly_flags = iso_anomaly_flags

print(f"Anomaly detection results:")
print(f"Isolation Forest anomalies: {iso_anomaly_flags.sum()} ({iso_anomaly_flags.mean()*100:.2f}%)")
print(f"Z-Score anomalies: {z_score_anomaly_flags.sum()} ({z_score_anomaly_flags.mean()*100:.2f}%)")
print(f"Final anomalies (Isolation Forest only): {final_anomaly_flags.sum()} ({final_anomaly_flags.mean()*100:.2f}%)")

# Create anomaly reasons
anomaly_reasons = []
for i in range(len(daily_profiles)):
    reasons = []
    if iso_anomaly_flags[i]:
        reasons.append(f"Isolation Forest (score: {iso_anomaly_scores[i]:.3f})")
    if peak_z_anomaly[i]:
        reasons.append(f"Peak Z-Score ({peak_z_scores[i]:.2f})")
    if mean_z_anomaly[i]:
        reasons.append(f"Mean Z-Score ({mean_z_scores[i]:.2f})")
    anomaly_reasons.append("; ".join(reasons) if reasons else "No anomaly")

print(f"Anomaly reasons generated for all days.")


Setting the final anomaly flag...
Anomaly detection results:
Isolation Forest anomalies: 60 (3.03%)
Z-Score anomalies: 0 (0.00%)
Final anomalies (Isolation Forest only): 60 (3.03%)
Anomaly reasons generated for all days.


In [7]:

# Prepare anomaly results dataframe
anomaly_results = pd.DataFrame({
    'date': daily_profiles['date'],
    'isolation_anomaly_score': iso_anomaly_scores,
    'isolation_anomaly_flag': iso_anomaly_flags,
    'peak_z_score': peak_z_scores,
    'mean_z_score': mean_z_scores,
    'z_score_anomaly_flag': z_score_anomaly_flags,
    'final_anomaly_flag': final_anomaly_flags,
    'anomaly_reason': anomaly_reasons
})

# Merge with clustering results
anomaly_results = anomaly_results.merge(
    clustering_results[['date', 'kmeans_cluster', 'dbscan_cluster', 'dbscan_noise_flag']],
    on='date',
    how='left'
)

# Add daily statistics for analysis
anomaly_results = anomaly_results.merge(
    daily_profiles[['date', 'daily_peak', 'daily_mean', 'daily_std']],
    on='date',
    how='left'
)

print(f"Anomaly results prepared: {anomaly_results.shape}")
print(f"\nSample of anomaly results:")
print(anomaly_results[['date', 'final_anomaly_flag', 'isolation_anomaly_score', 
                      'peak_z_score', 'mean_z_score', 'anomaly_reason']].head())


Anomaly results prepared: (1977, 14)

Sample of anomaly results:
         date  final_anomaly_flag  isolation_anomaly_score  peak_z_score  \
0  2020-01-01               False                -0.227287      0.283435   
1  2020-01-02               False                -0.213477      0.150268   
2  2020-01-03               False                -0.217256      0.232122   
3  2020-01-04               False                -0.150016      1.821206   
4  2020-01-05               False                -0.141816      2.317402   

   mean_z_score anomaly_reason  
0      0.170014     No anomaly  
1      0.170747     No anomaly  
2      0.167662     No anomaly  
3      1.843674     No anomaly  
4      1.904585     No anomaly  


In [8]:

# Save anomaly results
anomaly_results.to_csv(ANOMALY_RESULTS_PATH, index=False)
print(f"Anomaly results saved to: {ANOMALY_RESULTS_PATH}")

# Calculate and save anomaly metrics
total_days = len(anomaly_results)
anomaly_metrics = {
    'total_days': total_days,
    'anomaly_days': int(final_anomaly_flags.sum()),
    'anomaly_percentage': round(float(final_anomaly_flags.mean()) * 100, 2),
    'isolation_forest_anomaly_count': int(iso_anomaly_flags.sum()),
    'z_score_anomaly_count': int(z_score_anomaly_flags.sum()),
    'final_anomaly_count': int(final_anomaly_flags.sum()),
    'isolation_forest_metrics': {
        'contamination_setting': 0.03,
        'actual_contamination': round(float(iso_anomaly_flags.mean()), 4),
        'n_estimators': 300
    },
    'z_score_metrics': {
        'threshold': Z_SCORE_THRESHOLD,
        'peak_z_anomalies': int(peak_z_anomaly.sum()),
        'mean_z_anomalies': int(mean_z_anomaly.sum())
    },
    'data_info': {
        'date_range_start': daily_profiles['date'].min(),
        'date_range_end': daily_profiles['date'].max()
    }
}

# Save metrics
with open(MODELS_DIR / "anomaly_metrics.json", 'w') as f:
    json.dump(anomaly_metrics, f, indent=2)

print(f"Anomaly metrics saved to: {MODELS_DIR / 'anomaly_metrics.json'}")

print("\n=== ANOMALY DETECTION SUMMARY ===")
print(f"Total days analyzed: {total_days}")
print(f"Final anomalies detected: {anomaly_metrics['anomaly_days']} ({anomaly_metrics['anomaly_percentage']}%)")
print(f"Isolation Forest anomalies: {anomaly_metrics['isolation_forest_anomaly_count']}")
print(f"Z-Score anomalies: {anomaly_metrics['z_score_anomaly_count']}")
print(f"\nSaved files:")
print(f"- {MODELS_DIR / 'isolation_forest_model.pkl'}")
print(f"- {ANOMALY_RESULTS_PATH}")
print(f"- {MODELS_DIR / 'anomaly_metrics.json'}")


Anomaly results saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\anomaly_results.csv
Anomaly metrics saved to: D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\anomaly_metrics.json

=== ANOMALY DETECTION SUMMARY ===
Total days analyzed: 1977
Final anomalies detected: 60 (3.03%)
Isolation Forest anomalies: 60
Z-Score anomalies: 0

Saved files:
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\isolation_forest_model.pkl
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\anomaly_results.csv
- D:\Y4S1\Research\Component1-Behaviour-Intelligence\behaviour-intelligence\notebooks\outputs\models\anomaly_metrics.json


In [9]:
# Check against the app's anomaly results (read-only)
app_anomalies = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "anomaly_results.csv")

same_dates = list(app_anomalies['date']) == list(anomaly_results['date'])
same_flags = (app_anomalies['final_is_anomaly'].values == anomaly_results['final_anomaly_flag'].values).all()
same_scores = np.allclose(app_anomalies['isolation_anomaly_score'].values, anomaly_results['isolation_anomaly_score'].values)

print("=== CHECK AGAINST THE APP (data/processed/anomaly_results.csv) ===")
print(f"Anomaly days: notebook {int(anomaly_results['final_anomaly_flag'].sum())} | app {int(app_anomalies['final_is_anomaly'].sum())}")
print(f"Same dates: {same_dates}")
print(f"Same anomaly flag for every day: {same_flags}")
print(f"Same Isolation Forest scores: {same_scores}")

=== CHECK AGAINST THE APP (data/processed/anomaly_results.csv) ===
Anomaly days: notebook 60 | app 60
Same dates: True
Same anomaly flag for every day: True
Same Isolation Forest scores: True
